In [ ]:
# @title Mise en place de l'environnement
!rm -rf sample_data .config
!curl -sSL https://install.python-poetry.org | python -
path = %env PATH
poetry_path = "/root/.local/bin"
if poetry_path not in path:
    path = f"{poetry_path}:{path}"
    %set_env PATH $path
!poetry config virtualenvs.create false
!git config --global user.email "jeanne@durant.fr"
!git config --global user.name "Jeanne Durant"
!git config --global init.defaultBranch main
!git clone https://dagshub.com/m09/landscape-classifier.git
%cd /content/landscape-classifier
!pip install --quiet dvc mlflow
!dvc remote modify origin --local auth basic
%set_env MLFLOW_TRACKING_URI https://dagshub.com/m09/landscape-classifier.mlflow

# MLFlow, registre de modèles & déploiement

## Identifiants pour les serveurs DVC & MLFlow

Entrez ici vos identifiants DagsHub.

In [ ]:
username = "m09"
password = "542e264f9623e096e63c24e523c7edf30081de7f"

In [ ]:
# @title À exécuter
!dvc remote modify origin --local user {username}
!dvc remote modify origin --local password {password}
%set_env MLFLOW_TRACKING_USERNAME {username}
%set_env MLFLOW_TRACKING_PASSWORD {password}

## Récupération des données

In [ ]:
!wget https://github.com/m09/dataset-landscape/archive/refs/heads/main.zip
!unzip main.zip
!mv dataset-landscape-main data

## Publication d'un modèle sur un registre de modèle

Créez le fichier `train.py` pour entraîner un modèle en enregistrant les métriques avec [MLFlow Tracking](https://mlflow.org/docs/latest/tracking.html). Les pages sur [Keras](https://mlflow.org/docs/latest/python_api/mlflow.keras.html) de l'API Python MLFlow & de la [flavor Keras](https://mlflow.org/docs/latest/models.html#keras-keras) des modèles MLFlow pourront être utiles.

Publiez ensuite le modèle appris dans le registre de modèles MLFlow en fin d'exécution. [Cette page](https://mlflow.org/docs/latest/models.html#keras-keras) en parle.

Vous pourrez partir du code suivant&nbsp;:

```python
import tensorflow as tf

CLASS_NAMES = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
CLASS_INDICES = {l: i for i, l in enumerate(CLASS_NAMES)}


def get_images(
    dir_path: str, image_size: tuple[int, int], seed: int, shuffle: bool = True
) -> tuple[tf.data.Dataset, tf.data.Dataset]:
    return tf.keras.utils.image_dataset_from_directory(
        dir_path,
        class_names=CLASS_NAMES,
        batch_size=128,
        image_size=image_size,
        validation_split=0.3,
        subset="both",
        seed=seed,
    )


def get_lenet(
    image_size: tuple[int, int], learning_rate: float
) -> tf.keras.models.Model:
    def conv(filters: int, padding: str) -> tf.keras.layers.Conv2D:
        return tf.keras.layers.Conv2D(
            filters=filters, kernel_size=5, padding=padding, activation="sigmoid"
        )

    def pooling() -> tf.keras.layers.MaxPooling2D:
        return tf.keras.layers.MaxPooling2D()

    def dense(units: int, activation: str = "sigmoid") -> tf.keras.layers.Dense:
        return tf.keras.layers.Dense(units, activation=activation)

    model = tf.keras.Sequential(
        [
            tf.keras.layers.InputLayer(input_shape=(*image_size, 3)),
            conv(6, "same"),
            pooling(),
            conv(16, "valid"),
            pooling(),
            tf.keras.layers.Flatten(),
            dense(120),
            dense(84),
            dense(6, activation="softmax"),
        ],
        name="le_net",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


if __name__ == "__main__":
    train_dataset, val_dataset = get_images("data/seg_train", (100, 100), 42)
    model = get_lenet((100, 100), 0.0001)
    model.fit(train_dataset, validation_data=val_dataset, epochs=3)
    model.save("landscape_classifier.keras")
```

## Utilisation du registre de modèles MLFlow

En utilisant l'API MLFlow, effectuez les tâches suivantes sur le registre de modèles.

### Création d'un client MLFlow

Créez un client MLFlow et affichez l'adresse de tracking pour vérifier qu'elle est bien récupérée depuis la variable d'environnement définie en haut du notebook.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
import mlflow

client = mlflow.client.MlflowClient()
client.tracking_uri

### Recherche de modèle

Affichez tous les modèles enregistrés et pour chacun des modèles la version créée la plus récemment.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
def latest_version(latest_versions):
    return sorted(latest_versions, key=lambda v: -v.creation_timestamp)[0].version


for model in client.search_registered_models():
    print(model.name, latest_version(model.latest_versions))

### Récupération d'un modèle depuis le registre de modèle

Maintenant que vous savez récupérer depuis Python le nom et la version des modèles présents sur le registre, récupérez en un et vérifiez que c'est bien l'objet que vous anticipez.

Il y a plusieurs manières de charger un modèle si celui-ci dispose de plusieurs *flavors*. Qu'est-ce que cela change ?

In [ ]:
# Votre code ici

#### Solution

In [ ]:
import mlflow.keras


def uri_from_name_and_version(name: str, version: str | int) -> str:
    return f"models:/{name}/{version}"


uri = uri_from_name_and_version("lenet-landscape-classifier", 1)


model_keras = mlflow.keras.load_model(model_uri=uri)
model_keras.summary()
model = mlflow.pyfunc.load_model(model_uri=uri)
model

### Ajout d'alias et d'étiquettes

Les alias et les étiquettes (tags) permettent d'organiser les modèles dans le registre de modèles MLFLow. Les alias ciblent forcément une version en particulier alors que les étiquettes peuvent aussi cibler toutes les versions d'un modèle.

Ajoutez un alias `champion` à la dernière version de votre modèle, un tag `domain` à la valeur `cv` pour votre modèle et un tag `size` à la valeur `small` pour la dernière version de votre modèle.

In [ ]:
# Votre code ici

Note : la version installée de MLFlow sur DagsHub étant la 2.7.1 à l'heure de l'écriture de cette note, les alias ne sont pas visibles. Ils le seront par contre si vous utilisez une backend MLFlow à jour.

Récupérez maintenant votre modèle en utilisant l'alias plutôt que la version exacte.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
client.set_registered_model_alias("lenet-landscape-classifier", "champion", "1")
client.set_registered_model_tag("lenet-landscape-classifier", "domain", "cv")
client.set_model_version_tag("lenet-landscape-classifier", "1", "size", "small")

In [ ]:
import mlflow.keras


def uri_from_name_and_alias(name: str, alias: str | int) -> str:
    return f"models:/{name}@{alias}"


uri = uri_from_name_and_alias("lenet-landscape-classifier", "champion")


model_keras = mlflow.keras.load_model(model_uri=uri)
model_keras.summary()

### Changement de phase d'un modèle

Quand vous souhaitez promouvoir un modèle dans votre chaine de mise en production, la méthode recommandée actuellement est de copier le modèle vers un nouveau nom de modèle.

Copiez un modèle de votre choix vers un nouveau nom, qui sera l'ancien préfixé de `staging.`.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
client.copy_model_version(
    src_model_uri="models:/lenet-landscape-classifier@champion",
    dst_name="staging.lenet-landscape-classifier",
)

## Création d'une API avec FastAPI

Toutes les questions de cette partie sont à coder dans le fichier `landscape_classifier/api.py`.

### Création d'une fonction de prétraitement adaptée à l'inférence

Créez une fonction pour charger une image dans le même format que celui utilisé pendant l'entraînement. Quel est le point auquel il faut faire extrêmement attention à ce stade&nbsp;?


### Création d'une fonction de chargement de modèle

Créez une fonction qui récupère le modèle entraîné au préalable depuis le registre de modèle MLFlow. Cette fonction prendra en entrée l'URI du modèle à récupérer.

### Création d'une classe de retour FastAPI

FastAPI utilise l'excellente bibliothèque [Pydantic](https://docs.pydantic.dev/latest/) pour gérer les types d'entrée et de sortie des requêtes HTTP.

Créez une classe pour modéliser le type de retour avec la librairie Pydantic. La classe de retour devra contenir a minima la classe prédite (celle avec la probabilité maximale), et un dictionnaire des différentes classes et leur probabilité selon le modèle

### Création de l'API FastAPI

Implémentez l'API d'inférence à l'aide d'une méthode POST qui acceptera un fichier d'image. Vous pourrez vous aider de [cette page de documentation](https://fastapi.tiangolo.com/tutorial/request-files/).

## Création d'un Dockerfile

Écrivez un Dockerfile qui&nbsp;:

- Utilise l'image de base Python 3.10
- Copie un fichier `requirements.txt` pour installer des dépendances Python
- Installe ces dépendances
- Copie le fichier `api.py`
- Met en place un point d'entrée qui lance le serveur FastAPI

## Amélioration de l'API

Comment modifier l'API pour qu'elle soit plus adaptée à la mise en production par image Docker ?

*Votre réponse ici.*

## Solution

Pour toutes les questions qui n'ont pas de solution, consulter [le dépôt DagsHub](https://dagshub.com/m09/landscape-classifier/src/solution) de ces travaux pratiques sur sa branche `solution`.